In [26]:
import pandas as pd
import numpy as np

df = pd.read_csv("../../data/hc3_all.csv", header=0)

if "id" not in df.columns and "index" in df.columns:
    df = df.rename(columns={"index": "id"})

if "id" not in df.columns:
    raise ValueError("The raw HC3 dataset must contain an id or index column.")

if df["id"].isna().any() or df["id"].duplicated().any():
    df["id"] = range(len(df))

print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (24322, 5)


,id,question,human_answers,chatgpt_answers,source
0,0,"Why is every book I hear about a "" NY Times # 1 Best Seller "" ? ELI5 : Why is every book I hear about a "" NY Times # 1 Best Seller "" ? Should n't there only be one "" # 1 "" best seller ? Please explain like I'm five.","['Basically there are many categories of "" Best Seller "" . Replace "" Best Seller "" by something like "" Oscars "" and every "" best seller "" book is basically an "" oscar - winning "" book . May not have won the "" Best film "" , but even if you won the best director or best script , you \'re still an "" oscar - winning "" film . Same thing for best sellers . Also , IIRC the rankings change every week or something like that . Some you might not be best seller one week , but you may be the next week . I guess even if you do n\'t stay there for long , you still achieved the status . Hence , # 1 best seller .'\n ""If you 're hearing about it , it 's because it was a very good or very well - publicized book ( or both ) , and almost every good or well - publicized book will be # 1 on the NY Times bestseller list for at least a little bit . Kindof like how almost every big or good movies are # 1 at the box office on their opening weekend .""\n ""One reason is lots of catagories . However , how the NY Times calculates its best seller list is n't comprehensive , and is pretty well understood by publishers . So publishers can [ buy a few books ] ( URL_0 ) in the right bookstores and send a book to the top of the list for at least a week .""]","['There are many different best seller lists that are published by various organizations, and the New York Times is just one of them. The New York Times best seller list is a weekly list that ranks the best-selling books in the United States based on sales data from a number of different retailers. The list is published in the New York Times newspaper and is widely considered to be one of the most influential best seller lists in the book industry. \nIt\'s important to note that the New York Times best seller list is not the only best seller list out there, and there are many other lists that rank the top-selling books in different categories or in different countries. So it\'s possible that a book could be a best seller on one list but not on another. \nAdditionally, the term ""best seller"" is often used more broadly to refer to any book that is selling well, regardless of whether it is on a specific best seller list or not. So it\'s possible that you may hear about a book being a ""best seller"" even if it is not specifically ranked as a number one best seller on the New York Times list or any other list.']",reddit_eli5
1,1,"If salt is so bad for cars , why do we use it on the roads ? As the title states , why do we use it ? is there no other option or what ? Please explain like I'm five.","['salt is good for not dying in car crashes and car crashes are worse for cars then salt . Some places use other things , but salt is really cheap compared to most alternatives , although sand is pretty good .'\n 'In Minnesota and North Dakota , they tend to use sand , not salt . In these states , though , it is cold enough that the salt can not actually melt the snow , and so it refreezes and you end up with black ice on the roads , which is counterproductive . In other states whether there is snow but not so cold , wet asphalt is better than gritty snow for traction and so salt is favored for safety . Most people who live in states where salt is used , and by the way also in coastal cities , know that washing the car a couple times a week is critical .'\n ""Used to work in the salt industry ( yes , it 's a thing ) . Salt is one of the cheapest substances on earth . As in , you 'd be getting ripped off if you paid more than $ 150 per metric ton . ( In fact , the largest component of salt 's price is actually shipping . Yes , it costs more to transport salt than it does to produce it . ) Salt does its job ( lowering the freezing

## Find missing values

In [27]:
# Are there missing values?
print(df.isnull().values.any())
print(df.isnull().head())

nan_count = np.sum(df.isnull(), axis = 0)
print(f"Dataset NaN count:\n{nan_count}")


missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)
})

missing_summary

False
      id  question  human_answers  chatgpt_answers  source
0  False     False          False            False   False
1  False     False          False            False   False
2  False     False          False            False   False
3  False     False          False            False   False
4  False     False          False            False   False
Dataset NaN count:
id                 0
question           0
human_answers      0
chatgpt_answers    0
source             0
dtype: int64


,missing_count,missing_percent
id,0,0.0
question,0,0.0
human_answers,0,0.0
chatgpt_answers,0,0.0
source,0,0.0


We actually shouldn't need to correct any missing data in the source column as the data in that column are strings, and it doesn't include particularly important information for this project.

## Find duplicates

In [28]:
duplicate_count = df.duplicated().sum()

print(f"Duplicate rows: {duplicate_count}")

df[df.duplicated(keep=False)].sort_values(
    by=df.columns.tolist()
)

Duplicate rows: 0


,id,question,human_answers,chatgpt_answers,source


## Find inconsistencies

In [29]:
text_columns = df.select_dtypes(
    include=["object", "string"]
).columns

for col in text_columns:
    series = df[col].dropna().astype(str)

    whitespace_issues = series[
        series.str.strip() != series
    ]

    multiple_space_issues = series[
        series.str.contains(r"\s{2,}", regex=True)
    ]

    print(f"\n{col} Column:")
    print(f"Leading/trailing whitespace: {len(whitespace_issues)}")
    print(f"Multiple spaces: {len(multiple_space_issues)}")


question Column:
Leading/trailing whitespace: 1944
Multiple spaces: 213

human_answers Column:
Leading/trailing whitespace: 0
Multiple spaces: 19936

chatgpt_answers Column:
Leading/trailing whitespace: 0
Multiple spaces: 1843

source Column:
Leading/trailing whitespace: 0
Multiple spaces: 0


### Check for empty lists

In [30]:
print("Empty human answer lists:")
print((df["human_answers"] == "[]").sum())

print("\nEmpty ChatGPT answer lists:")
print((df["chatgpt_answers"] == "[]").sum())

Empty human answer lists:
0

Empty ChatGPT answer lists:
455


### Remove rows without a ChatGPT answer

In [31]:
df = df[df["chatgpt_answers"] != "[]"].copy()

print("Rows after removing empty ChatGPT answers:", len(df))

Rows after removing empty ChatGPT answers: 23867


### Verify no empty lists

In [32]:
print("Empty human answer lists:",
      (df["human_answers"] == "[]").sum())

print("Empty ChatGPT answer lists:",
      (df["chatgpt_answers"] == "[]").sum())

Empty human answer lists: 0
Empty ChatGPT answer lists: 0


## Create the cleaned dataset

In [33]:
clean = df.copy()


# Clean text
text_columns = clean.select_dtypes(
    include=["object", "string"]
).columns

for col in text_columns:
    clean[col] = (
        clean[col]
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

## Create the expanded answer-level dataset

The cleaned dataset remains one row per question for leakage-safe splitting.

This separate dataset expands the human and ChatGPT answer lists into one labeled row per answer.

In [34]:
import ast
import re
import unicodedata

def parse_answer_list(value):
    if isinstance(value, list):
        return value

    repaired = re.sub(r"(?<=[\x27\x22])\s+(?=[\x27\x22])", ", ", value)
    return ast.literal_eval(repaired)

def normalize_text(text):
    if not isinstance(text, str):
        return text

    # Normalize Unicode characters
    text = unicodedata.normalize("NFKC", text)

    # Standardize curly quotes/apostrophes
    text = (
        text
        .replace("’", "'")
        .replace("‘", "'")
        .replace("“", '"')
        .replace("”", '"')
    )

    # Collapse repeated whitespace
    text = re.sub(r"\s+", " ", text)

    # Fix spaces immediately around apostrophes
    # don ' t -> don't
    text = re.sub(r"\s*'\s*", "'", text)

    # Fix tokenized contractions
    # do n't -> don't
    # I 'm -> I'm
    # you 're -> you're
    text = re.sub(
        r"\b(\w+)\s+(n't|'s|'m|'re|'ve|'ll|'d)\b",
        r"\1\2",
        text,
        flags=re.IGNORECASE
    )

    # Remove spaces before punctuation
    # word . -> word.
    text = re.sub(r"\s+([.,!?;:%])", r"\1", text)

    return text.strip()

parts = []

for column, label in [("human_answers", "human"), ("chatgpt_answers", "chatgpt")]:
    part = clean[["id", "question", "source", column]].copy()
    part[column] = part[column].apply(parse_answer_list)
    part = part.explode(column).rename(columns={column: "text"})
    part["label"] = label
    part["target"] = part["label"].map({"human": 0, "chatgpt": 1})
    parts.append(part)

expanded = pd.concat(parts, ignore_index=True)

expanded = expanded[expanded["text"].notna()].copy()

In [35]:
# Temporary audit copy
expanded["original_text"] = expanded["text"]

# Normalize answer text
expanded["text"] = expanded["text"].apply(normalize_text)

# Remove empty answers
expanded = (expanded[expanded["text"].str.strip().ne("")].reset_index(drop=True))

# Inspect what changed
changed = expanded[expanded["original_text"] != expanded["text"]]

print(
    f"Normalized {len(changed):,} "
    f"of {len(expanded):,} answers."
)

display(changed[["source","label","original_text","text"]].head(50))

# Remove audit column before saving
expanded = expanded.drop(columns=["original_text"])

Normalized 60,381 of 84,066 answers.


,source,label,original_text,text
0,reddit_eli5,human,"Basically there are many categories of "" Best Seller "" . Replace "" Best Seller "" by something like "" Oscars "" and every "" best seller "" book is basically an "" oscar - winning "" book . May not have won the "" Best film "" , but even if you won the best director or best script , you 're still an "" oscar - winning "" film . Same thing for best sellers . Also , IIRC the rankings change every week or something like that . Some you might not be best seller one week , but you may be the next week . I guess even if you do n't stay there for long , you still achieved the status . Hence , # 1 best seller .","Basically there are many categories of "" Best Seller "". Replace "" Best Seller "" by something like "" Oscars "" and every "" best seller "" book is basically an "" oscar - winning "" book. May not have won the "" Best film "", but even if you won the best director or best script, you're still an "" oscar - winning "" film. Same thing for best sellers. Also, IIRC the rankings change every week or something like that. Some you might not be best seller one week, but you may be the next week. I guess even if you don't stay there for long, you still achieved the status. Hence, # 1 best seller."
1,reddit_eli5,human,"If you 're hearing about it , it 's because it was a very good or very well - publicized book ( or both ) , and almost every good or well - publicized book will be # 1 on the NY Times bestseller list for at least a little bit . Kindof like how almost every big or good movies are # 1 at the box office on their opening weekend .","If you're hearing about it, it's because it was a very good or very well - publicized book ( or both ), and almost every good or well - publicized book will be # 1 on the NY Times bestseller list for at least a little bit. Kindof like how almost every big or good movies are # 1 at the box office on their opening weekend."
2,reddit_eli5,human,"One reason is lots of catagories . However , how the NY Times calculates its best seller list is n't comprehensive , and is pretty well understood by publishers . So publishers can [ buy a few books ] ( URL_0 ) in the right bookstores and send a book to the top of the list for at least a week .","One reason is lots of catagories. However, how the NY Times calculates its best seller list isn't comprehensive, and is pretty well understood by publishers. So publishers can [ buy a few books ] ( URL_0 ) in the right bookstores and send a book to the top of the list for at least a week."
3,reddit_eli5,human,"salt is good for not dying in car crashes and car crashes are worse for cars then salt . Some places use other things , but salt is really cheap compared to most alternatives , although sand is pretty good .","salt is good for not dying in car crashes and car crashes are worse for cars then salt. Some places use other things, but salt is really cheap compared to most alternatives, although sand is pretty good."
4,reddit_eli5,human,"In Minnesota and North Dakota , they tend to use sand , not salt . In these states , though , it is cold enough that the salt can not actually melt the snow , and so it refreezes and you end up with black ice on the roads , which is counterproductive . In other states whether there is snow but not so cold , wet asphalt is better than gritty snow for traction and so salt is favored for safety . Most people who live in states where salt is used , and by the way also in coastal cities , know that washing the car a couple times a week is critical .","In Minnesota and North Dakota, they tend to use sand, not salt. In these states, though, it is cold enough that the salt can not actually melt the snow, and so it refreezes and you end up with black ice on the roads, which is counterproductive. In other states whether there is snow but not so cold, wet asphalt is better than gritty snow for traction and so salt is favored for safety. Most people who live in states where salt is u

## Saving the dataset

In [36]:
expanded_path = "../../data/processed/expanded_dataset.csv"
expanded.to_csv(expanded_path, index=False, chunksize=1000)

print(expanded["label"].value_counts())

print(f"Saved cleaned dataset to: {expanded_path}")
print(f"Original shape: {df.shape}")
print(f"Cleaned shape: {expanded.shape}")

label
human      57181
chatgpt    26885
Name: count, dtype: int64
Saved cleaned dataset to: ../../data/processed/expanded_dataset.csv
Original shape: (23867, 5)
Cleaned shape: (84066, 6)
